In [154]:
import os
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from urllib.parse import quote_plus

def get_html(driver, job_name, page_number):
    encoded_job_name = quote_plus(job_name)
    url = f'https://nofluffjobs.com/?criteria=jobPosition%3D%27{encoded_job_name}%27&page={page_number}'
    driver.get(url)
    html_content = driver.page_source
    return html_content

def save_html_to_file(html_content, job_name, page_number):
    filename = f'/Users/katka/Desktop/project_folder/data/raw/{job_name}_{page_number}.html'
    with open(filename, 'w', encoding='utf-8') as file:
        file.write(html_content)

driver = webdriver.Chrome()

def retrieve_and_save_first_page(driver, job_name, page_number):
    html_content = get_html(driver, job_name, page_number)
    save_html_to_file(html_content, job_name, page_number)

retrieve_and_save_first_page(driver, 'data scientist', 10)
retrieve_and_save_first_page(driver, 'data analyst', 10)
retrieve_and_save_first_page(driver, 'data engineer', 10)

driver.close()
driver.quit()


In [185]:
import os
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
import re


roles = ['engineer', 'scientist', 'analyst']

raw_data_dir = '/Users/katka/Desktop/project_folder/data/raw'
interim_data_dir = '/Users/katka/Desktop/project_folder/data/interim'

dataframes = {}

for role in roles:
    job_ads_list = []
    output_file = interim_data_dir + f'/job_offers_{role}_{datetime.today().strftime("%Y_%m_%d")}.csv'

    for filename in os.listdir(raw_data_dir):
        if role in filename and filename.endswith('.html'):
            file_path = os.path.join(raw_data_dir, filename)
            with open(file_path, 'r', encoding='utf-8') as file:
                html_code = file.read()
                single_ads = extract_single_ads(html_code)
                for ad in single_ads:
                    ad_info = parse_single_ad(str(ad), role)

                    if role.lower() == 'analyst':
                        if 'analyst' in ad_info['name'].lower() and 'data' in ad_info['name'].lower():
                            job_ads_list.append(ad_info)
                    elif role.lower() == 'engineer':
                        if 'engineer' in ad_info['name'].lower() and 'data' in ad_info['name'].lower():
                            job_ads_list.append(ad_info)
                    else:
                        if role.lower() in ad_info['name'].lower():
                            job_ads_list.append(ad_info)

    dataframes[f'df_{role}'] = pd.DataFrame(job_ads_list)

df_engineer = dataframes['df_engineer']
df_scientist = dataframes['df_scientist']
df_analyst = dataframes['df_analyst']

df_engineer.to_csv(interim_data_dir + '/job_offers_engineer.csv', sep=';', encoding='utf-8', index=False)
df_scientist.to_csv(interim_data_dir + '/job_offers_scientist.csv', sep=';', encoding='utf-8', index=False)
df_analyst.to_csv(interim_data_dir + '/job_offers_analyst.csv', sep=';', encoding='utf-8', index=False)


In [114]:
file_path = '/Users/katka/Desktop/project_folder/data/raw/data analyst_1.html'

with open(file_path, 'r', encoding='utf-8') as file:
    html_content = file.read()

soup = BeautifulSoup(html_content, 'html.parser')

print(soup.prettify())



FileNotFoundError: [Errno 2] No such file or directory: '/Users/katka/Desktop/project_folder/data/raw/data analyst_1.html'

DataFrame Shape: (20, 6)
                                     name               company  \
0                   Senior Data Scientist          Reply Polska   
1  Data Scientist, Production Engineering    Brainly Sp. z o.o.   
2                   Junior Data Scientist  datanalytics DA GmbH   
3                          Data Scientist                Devire   
4             Data Scientist/Data Analyst          Object First   

                           technology             job  \
0   AI, Python, Machine learning, AWS  data scientist   
1  Data, Python, NLP, Computer vision  data scientist   
2   AI, Python, Git, Machine learning  data scientist   
3              AI, Python, SQL, T-SQL  data scientist   
4     Data, Python, Power BI, Jupyter  data scientist   

                                 location  \
0  {'city': 'Katowice', 'country': 'N/A'}   
1    {'city': 'Remote', 'country': 'N/A'}   
2    {'city': 'Remote', 'country': 'N/A'}   
3    {'city': 'Gdynia', 'country': 'N/A'}   
4   

In [186]:
import pandas as pd
import re

def clean_and_transform(df):
    cleaned_df = df.copy()

    cleaned_df['location_city'] = cleaned_df['location'].apply(lambda x: x.get('city', ''))

    # Manual city corrections
    city_corrections = {
        'wroclove': 'wroclaw',
        'wroclaw': 'wroclaw',
        'krakow': 'krakow',
        'kraków +1': 'krakow',
        'kraków +2': 'krakow',
        'warsaw':'warszawa'
    }
    cleaned_df['location_city'] = cleaned_df['location_city'].replace(city_corrections)

    cleaned_df['location_city'] = cleaned_df['location_city'].str.lower()
    cleaned_df['location_city'] = cleaned_df['location_city'].apply(lambda x: 'N/A' if x == 'zdalnie' or x == 'remote' else x)

    cleaned_df = cleaned_df[cleaned_df['salary'].apply(lambda x: 'PLN' in x.get('currency', ''))]

    cleaned_df['salary_low'] = cleaned_df['salary'].apply(lambda x: int(x.get('low')) if x.get('low') is not None else None)
    cleaned_df['salary_high'] = cleaned_df['salary'].apply(lambda x: int(x.get('high')) if x.get('high') is not None else None)

    cleaned_df['salary_avg'] = cleaned_df.apply(lambda row: (row['salary_low'] + row['salary_high']) / 2 if row['salary_low'] is not None and row['salary_high'] is not None else None, axis=1)

    cleaned_df['name'] = cleaned_df['name'].str.lower()

    cleaned_df['is_senior'] = cleaned_df['name'].apply(lambda x: 1 if 'senior' in x.lower() else 0)

    cleaned_df.drop(columns=['salary', 'location'], inplace=True)

    return cleaned_df

df_engineer_cleaned = clean_and_transform(df_engineer)
df_scientist_cleaned = clean_and_transform(df_scientist)
df_analyst_cleaned = clean_and_transform(df_analyst)

output_folder = '/Users/katka/Desktop/project_folder/data/pro/'

df_engineer_cleaned.to_csv(output_folder + 'job_offers_engineer_cleaned.csv', sep=';', encoding='utf-8', index=False)
df_scientist_cleaned.to_csv(output_folder + 'job_offers_scientist_cleaned.csv', sep=';', encoding='utf-8', index=False)
df_analyst_cleaned.to_csv(output_folder + 'job_offers_analyst_cleaned.csv', sep=';', encoding='utf-8', index=False)


In [167]:
cleaned_df.tail()

,name,company,technology,job,location_city,salary_low,salary_high,salary_avg,is_senior
14,data analyst (sap),Team Connect Sp. z o.o.,"Data, SAP, Data models, SAP BW",data analyst,N/A,23500,28500,26000.0,0
15,expert data analyst,emagine Sp. Z o.o.,"Data, Business Analyst, SQL, SAS",data analyst,gdańsk,25200,28560,26880.0,0
16,python data analyst / programmer,Polskie Technologie,"Data, SQL, SQL Server, Python",data analyst,warszawa,8700,15000,11850.0,0
17,medior/senior business data analyst / bi engineer,NIX Tech Kft.,"Business Analysis, Hungarian Language, DWH, SQL",data analyst,budapest,8856,12178,10517.0,1
18,senior data analyst,Devire,"Data, SQL, Power BI, Tableau",data analyst,kraków,17000,20000,18500.0,1


In [176]:
df_scientist_cleaned.head()

,name,company,technology,job,location_city,salary_low,salary_high,salary_avg,is_senior
0,data scientist/data analyst,Object First,"Data, Python, Power BI, Jupyter",data scientist,N/A,24000,36000,30000.0,0
1,senior data scientist,Reply Polska,"AI, Python, Machine learning, AWS",data scientist,katowice,16000,22000,19000.0,1
2,"data scientist, production engineering",Brainly Sp. z o.o.,"Data, Python, NLP, Computer vision",data scientist,N/A,17500,30000,23750.0,0
3,junior data scientist,datanalytics DA GmbH,"AI, Python, Git, Machine learning",data scientist,N/A,28941,39794,34367.5,0
4,data scientist,Devire,"AI, Python, SQL, T-SQL",data scientist,gdynia,16800,21840,19320.0,0


# question 1


In [187]:
num_jobs_engineer = df_engineer_cleaned.shape[0]
num_jobs_scientist = df_scientist_cleaned.shape[0]
num_jobs_analyst = df_analyst_cleaned.shape[0]

print(f"Number of job offers for Data Engineer: {num_jobs_engineer}")
print(f"Number of job offers for Data Scientist: {num_jobs_scientist}")
print(f"Number of job offers for Data Analyst: {num_jobs_analyst}")


Number of job offers for Data Engineer: 65
Number of job offers for Data Scientist: 22
Number of job offers for Data Analyst: 16


# question 2

In [188]:
avg_salary_engineer = df_engineer_cleaned['salary_avg'].mean()
avg_salary_scientist = df_scientist_cleaned['salary_avg'].mean()
avg_salary_analyst = df_analyst_cleaned['salary_avg'].mean()

print(f"Average salary for Data Engineer: {avg_salary_engineer:.2f} PLN")
print(f"Average salary for Data Scientist: {avg_salary_scientist:.2f} PLN")
print(f"Average salary for Data Analyst: {avg_salary_analyst:.2f} PLN")


Average salary for Data Engineer: 22427.18 PLN
Average salary for Data Scientist: 22698.30 PLN
Average salary for Data Analyst: 20679.44 PLN


# question 3

In [190]:
combined_df = pd.concat([df_engineer_cleaned, df_scientist_cleaned, df_analyst_cleaned])

city_job_counts = combined_df[combined_df['location_city'] != 'N/A']['location_city'].value_counts()

top_5_cities = city_job_counts.head(5)

print("Top 5 cities with the most job offers:")
print(top_5_cities)


Top 5 cities with the most job offers:
location_city
warszawa    16
kraków      10
budapest    10
warsaw       5
katowice     3
Name: count, dtype: int64


# question 4

In [191]:

average_salary_per_city = combined_df[combined_df['location_city'] != 'N/A'].groupby('location_city')['salary_avg'].mean()


highest_salary_city = average_salary_per_city.idxmax()
highest_salary_amount = average_salary_per_city.max()


print(f"The highest average salary is offered in {highest_salary_city}, with an average of {highest_salary_amount:.2f} PLN.")


The highest average salary is offered in gdańsk, with an average of 29986.00 PLN.


# question 5

In [195]:
def calculate_salary_difference(df):

    avg_salary_regular = df[df['is_senior'] == 0]['salary_avg'].mean()


    avg_salary_senior = df[df['is_senior'] == 1]['salary_avg'].mean()

    salary_difference = avg_salary_senior - avg_salary_regular

    return avg_salary_regular, avg_salary_senior, salary_difference

avg_reg_engineer, avg_senior_engineer, diff_engineer = calculate_salary_difference(df_engineer_cleaned)
print(f"Data Engineer - Avg Regular Salary: {avg_reg_engineer}, Avg Senior Salary: {avg_senior_engineer}, Salary Difference: {diff_engineer}")


avg_reg_scientist, avg_senior_scientist, diff_scientist = calculate_salary_difference(df_scientist_cleaned)
print(f"Data Scientist - Avg Regular Salary: {avg_reg_scientist}, Avg Senior Salary: {avg_senior_scientist}, Salary Difference: {diff_scientist}")


avg_reg_analyst, avg_senior_analyst, diff_analyst = calculate_salary_difference(df_analyst_cleaned)
print(f"Data Analyst - Avg Regular Salary: {avg_reg_analyst}, Avg Senior Salary: {avg_senior_analyst}, Salary Difference: {diff_analyst}")


Data Engineer - Avg Regular Salary: 22347.132075471698, Avg Senior Salary: 22780.75, Salary Difference: 433.61792452830196
Data Scientist - Avg Regular Salary: 21468.55, Avg Senior Salary: 23723.083333333332, Salary Difference: 2254.533333333333
Data Analyst - Avg Regular Salary: 21532.045454545456, Avg Senior Salary: 18803.7, Salary Difference: -2728.345454545455


# question 6

In [196]:

min_salary_engineer = df_engineer_cleaned['salary_low'].min()
min_salary_scientist = df_scientist_cleaned['salary_low'].min()
min_salary_analyst = df_analyst_cleaned['salary_low'].min()


total_min_cost = min_salary_engineer + min_salary_scientist + min_salary_analyst

print(f"Minimum salary for Data Engineer: {min_salary_engineer}")
print(f"Minimum salary for Data Scientist: {min_salary_scientist}")
print(f"Minimum salary for Data Analyst: {min_salary_analyst}")
print(f"Total minimum cost for hiring one of each: {total_min_cost}")


Minimum salary for Data Engineer: 6089
Minimum salary for Data Scientist: 6642
Minimum salary for Data Analyst: 6642
Total minimum cost for hiring one of each: 19373


# question 7

In [197]:
import pandas as pd


combined_df = pd.concat([df_engineer_cleaned, df_scientist_cleaned, df_analyst_cleaned])

combined_df = combined_df[combined_df['location_city'] != 'N/A']

cities_with_all_positions = combined_df.groupby('location_city').apply(lambda x: all(job in x['job'].values for job in ['data engineer', 'data scientist', 'data analyst']))

valid_cities = cities_with_all_positions[cities_with_all_positions].index


min_cost_per_city = combined_df[combined_df['location_city'].isin(valid_cities)].groupby('location_city').apply(lambda x: x.groupby('job')['salary_low'].min().sum())


cheapest_city = min_cost_per_city.idxmin()
cheapest_cost = min_cost_per_city.min()

print(f"The cheapest city to form the team is {cheapest_city} with a total minimum cost of {cheapest_cost}")


The cheapest city to form the team is budapest with a total minimum cost of 19373
